#Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

In [0]:
df.display()

In [0]:
df.printSchema()


#Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Normalization


In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

##Clean Cost

In [0]:
df = df.withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))

##Split Product Key

In [0]:
%sql
select *
from workspace.bronze.erp_px_cat_g1v2

In [0]:
#Split product key to get category 
df = df.withColumn("prd_cat", F.substring("prd_key", 1, 5))
df = df.withColumn("prd_key", F.substring("prd_key", 7, F.length("prd_key")))
                   
#Change "-" to "_"
df = df.withColumn("prd_cat", F.regexp_replace("prd_cat", "-", "_"))
display(df)
            

## Rename Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_cat": "product_catagory",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)



## Sanity checks of dataframe


In [0]:
df.limit(10).display()

#Writing into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity checks of silver table


In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10